# Python Script to Run the Multiple-Node Colombia Case Study

### 1. Importing calliope model and built-in cal-graph module for post-process analysis

In [ ]:
import calliope

### 3. Assigning a name to the model settings file and run it

In [ ]:
model = calliope.Model('biomass_model.yaml')
model.run()

### 4. Saving the input data and output results in separated csv files

In [ ]:
model.to_csv(r'Results\solarFarm_cons_test1')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### 5. Create graphs

In [ ]:
model.results

In [ ]:
model.results['cost_investment']

In [ ]:
## Only inputs or only results
#
# model.plot.capacity(array='inputs')
model.plot.timeseries(array='results')
#model.plot.summary()

## Only consumed resource
#model.plot.timeseries(array='resource_con')

# ## Only consumed resource and 'power' carrier flow
# model.plot.timeseries(array=['power', 'resource_con'])

# model.plot.timeseries(subset={'tech_groups': [],'costs': ['monetary']})
# model.plot.timeseries(subset={'costs': ['monetary']})
# model.plot.timeseries(subset={'techs': [' pv_large_scale','on_wind_pp','bio_pp'],'costs': ['monetary']})

## test results

In [ ]:
# After model.run()
res = model.results  # xarray.Dataset
"Objective:", res.attrs.get("objective_function_value")

In [ ]:
# 2) Exported energy (sum over all exporting techs & hours)
# In 0.6.x the variable exists only if export is enabled and used.
if "carrier_export" in res:
    exp_sum = res["carrier_export"].sum().item()  # kWh
    print("Total exported energy [kWh]:", exp_sum)
    # By tech / location (if you want the breakdown)
    by_loc_tech = res["carrier_export"].sum("timesteps")
    print(by_loc_tech.to_series().sort_values(ascending=False).head())
else:
    print("No 'carrier_export' variable in results (export not enabled/used).")


In [ ]:
# 3) Money side: costs/revenues
# Calliope tracks costs by 'costs' dimension (e.g., 'energy_cap','storage_cap','om_prod','export', ...).
if "cost" in res:
    cost = res["cost"]  # €/year (annuitized), summed over timesteps for variable parts
    # Total monetary cost (incl. revenues)
    total_cost = cost.sum().item()
    print("Total monetary cost [€]:", total_cost)

    # Export *revenue* specifically (negative values)
    if "export" in cost.coords.get("costs", []):
        export_revenue = cost.sel(costs="export").sum().item()
        print("Export revenue (negative = revenue) [€]:", export_revenue)


## RoI

In [ ]:
def crf(r, n):
    return (r*(1+r)**n)/(((1+r)**n)-1)


In [ ]:
model_baseline= calliope.Model('baseline_model_export.yaml')
model_baseline.run()

In [ ]:
model_baseline.to_csv(r'Results\baseline_export_test1')

In [ ]:
# After you run baseline:
resB = model_baseline.results
costB = resB["cost"].sum().item()
"cost", costB

In [ ]:
# After you run project:
resP = model.results
costP = resP["cost"].sum().item()
print(resP["cost"].dims)                      # should be ('costs','loc_techs_cost')
print(resP["cost"].coords["costs"].values)    # what labels exist?
print(resP["cost"].coords["loc_techs_cost"].values)  # which techs?

In [ ]:



S = costB - costP  # annual savings (€/yr)
annualized_roi = S / capex_ann if capex_ann != 0 else float("nan")

# If using one r,n across techs:
r = 0.06   # real WACC
n = 15     # years
capex0 = capex_ann / crf(r, n)
payback_years = capex0 / S if S > 0 else float("inf")

print(f"Annual savings S: {S:,.0f} €/yr")
print(f"Annualized ROI: {100*annualized_roi:.1f}%")
print(f"Simple payback: {payback_years:.1f} years")

# Optional: NPV / IRR
import numpy as np
years = n
cashflows = [-capex0] + [S]*years
npv = sum(cf/((1+r)**t) for t,cf in enumerate(cashflows))
irr = np.irr(cashflows)
print(f"NPV (real €): {npv:,.0f}")
print(f"IRR: {100*irr:.1f}%")


In [ ]:
import numpy as np

# ---------- Helpers ----------
def crf(r, n):
    """Capital Recovery Factor for real discount rate r and lifetime n (years)."""
    return (r * (1 + r) ** n) / ((1 + r) ** n - 1)

def irr_bisection(cashflows, lo=-0.99, hi=1.0, tol=1e-6, max_iter=200):
    """IRR via bisection (no external deps). Returns np.nan if no sign change."""
    # NPV function
    def npv(rate):
        return sum(cf / ((1 + rate) ** t) for t, cf in enumerate(cashflows))
    if npv(lo) * npv(hi) > 0:
        return np.nan
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        v = npv(mid)
        if abs(v) < tol:
            return mid
        if npv(lo) * v < 0:
            hi = mid
        else:
            lo = mid
    return mid

# ---------- Path A: Using LCOE ----------
def roi_from_lcoe(
    lcoe_eur_per_kwh,
    e_self_kwh, e_export_kwh,
    p_import_eur_per_kwh, p_export_eur_per_kwh,
    r=None, n=None,
    fom_eur_per_kwyr=None,  # optional (if you want payback from LCOE)
    vom_eur_per_kwh=0.0,    # optional
    annual_energy_total_kwh=None  # if None, uses e_self+e_export
):
    """
    Returns dict with:
      S (annual savings), annualized_roi, capex0 (if derivable), payback_years (if capex0 & S>0), IRR (if r,n & capex0)
    """
    E_self = float(e_self_kwh)
    E_exp  = float(e_export_kwh)
    E_tot  = float(annual_energy_total_kwh) if annual_energy_total_kwh is not None else (E_self + E_exp)

    # Annualized total cost of the asset from LCOE:
    annualized_cost = lcoe_eur_per_kwh * E_tot

    # Annual "value" of energy (import avoided + export revenue):
    value = (p_import_eur_per_kwh * E_self) + (p_export_eur_per_kwh * E_exp)

    # Annual savings / profit vs buying all from grid (and not exporting):
    S = value - annualized_cost

    # Annualized ROI vs annualized cost
    annualized_roi = S / annualized_cost if annualized_cost != 0 else np.nan

    out = {
        "annual_savings_eur_per_yr": S,
        "annualized_roi_pct": 100 * annualized_roi if np.isfinite(annualized_roi) else np.nan,
        "annualized_cost_eur_per_yr": annualized_cost,
    }

    # Optional: back out initial CAPEX from LCOE if r,n and (FOM/VOM) are known/assumed.
    # LCOE ≈ (CRF*CAPEX + FOM) / E_annual + VOM
    # -> CAPEX ≈ ((LCOE - VOM) * E_annual - FOM) / CRF
    if (r is not None) and (n is not None):
        CRF = crf(r, n)
        FOM = 0.0 if fom_eur_per_kwyr is None else float(fom_eur_per_kwyr)
        # If you gave FOM per kW·yr, you need installed kW to convert to €/yr.
        # If you don't have kW, set FOM=0 or precompute FOM €/yr outside and pass it here.

        capex0 = ((lcoe_eur_per_kwh - vom_eur_per_kwh) * E_tot - FOM) / CRF
        out["capex0_eur"] = capex0 if np.isfinite(capex0) else np.nan

        payback = (capex0 / S) if (S > 0) else np.inf
        out["simple_payback_years"] = payback

        # IRR from a flat cashflow: Year0=-capex0, Years 1..n: +S
        cashflows = [-capex0] + [S] * int(n)
        out["irr_pct"] = 100 * irr_bisection(cashflows)

    return out

# ---------- Path B: Using Calliope results ----------
def roi_from_calliope(res_baseline, res_project, r=None, n=None):
    """
    Compute savings, annualized ROI, and (optionally) payback/IRR from Calliope results.
    Expects res_* are xarray.Datasets from model.results
    """
    costB = float(res_baseline["cost"].sum().item())
    costP = float(res_project["cost"].sum().item())
    S = costB - costP  # €/yr

    out = {
        "annual_savings_eur_per_yr": S,
        "baseline_total_cost_eur_per_yr": costB,
        "project_total_cost_eur_per_yr": costP,
    }

    # Try to extract annuitized capex (if cost classes available)
    costs_labels = set(res_project["cost"].coords["costs"].values.tolist())
    if {"energy_cap", "storage_cap"}.issubset(costs_labels):
        capex_ann = float(res_project["cost"].sel(costs=["energy_cap", "storage_cap"]).sum().item())
        out["capex_ann_eur_per_yr"] = capex_ann
        out["annualized_roi_pct"] = 100 * (S / capex_ann) if capex_ann else np.nan
        # De-annualize initial CAPEX if r,n provided
        if (r is not None) and (n is not None):
            CRF = crf(r, n)
            capex0 = capex_ann / CRF
            out["capex0_eur"] = capex0
            out["simple_payback_years"] = (capex0 / S) if (S > 0) else np.inf
            cashflows = [-capex0] + [S] * int(n)
            out["irr_pct"] = 100 * irr_bisection(cashflows)
    else:
        # No breakdown → still return annualized ROI vs total annual cost (rough)
        out["capex_ann_eur_per_yr"] = np.nan
        out["annualized_roi_pct"] = 100 * (S / costP) if costP else np.nan
        # For payback/IRR you need initial CAPEX; either re-run with cost classes
        # or reconstruct CAPEX from your installed kW/kWh * input unit costs.

    return out

# ---------- Example usage ----------
if __name__ == "__main__":
    # A) From LCOE:
    lcoe = 0.085   # €/kWh
    P_imp = 0.25   # €/kWh
    P_exp = 0.08   # €/kWh (FiT or wholesale net of fees)
    E_self = 1_200_000   # kWh/yr
    E_exp  =   300_000   # kWh/yr
    r, n = 0.06, 15
    # If you don't know FOM/VOM, leave them default; you'll get annualized ROI; payback uses a derived CAPEX.
    result_A = roi_from_lcoe(lcoe, E_self, E_exp, P_imp, P_exp, r=r, n=n)
    print("From LCOE:", result_A)
    print("------------------------")

    # B) From Calliope:
    res_baseline = model_baseline.results
    res_project  = model.results
    result_B = roi_from_calliope(res_baseline, res_project, r=0.06, n=15)
    print("From Calliope:", result_B)
